# Day 09. Exercise 00
# Regularization

## 0. Imports

In [211]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from sklearn.svm import SVC

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

import joblib
from sklearn.metrics import confusion_matrix

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [212]:
df = pd.read_csv('../data/dayofweek.csv')

In [213]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [214]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [215]:
X_test.shape

(338, 43)

In [216]:
X_train.shape

(1348, 43)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [217]:
%%time
lr = LogisticRegression(random_state=21, fit_intercept=False)

skf = StratifiedKFold(n_splits=10)

valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    lr.fit(X_train_fold, y_train_fold)

    train_pred = lr.predict(X_train_fold)
    valid_pred = lr.predict(X_valid_fold)

    train_acc = accuracy_score(y_train_fold, train_pred)
    valid_acc = accuracy_score(y_valid_fold, valid_pred)

    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.62819   |   valid -  0.59259
train -  0.64716   |   valid -  0.62963
train -  0.63479   |   valid -  0.57037
train -  0.65540   |   valid -  0.61481
train -  0.63314   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63591   |   valid -  0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: total: 312 ms
Wall time: 298 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [218]:
%%time
lr_none = LogisticRegression(penalty=None, solver='saga', random_state=21, fit_intercept=False, max_iter=5000)
skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    lr_none.fit(X_train_fold, y_train_fold) 
    valid_scores.append(accuracy_score((y_valid_fold), lr_none.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was depreca

Average accuracy on crossval is 0.62462
Std is 0.03379
CPU times: total: 22.3 s
Wall time: 22.9 s


In [219]:
%%time
lr_l1 = LogisticRegression(penalty='l1', solver='saga', random_state=21, fit_intercept=False, max_iter=5000)
skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    lr_l1.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, lr_l1.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2

Average accuracy on crossval is 0.59646
Std is 0.02848
CPU times: total: 3.81 s
Wall time: 3.88 s


In [220]:
%%time
lr_l2 = LogisticRegression(penalty='l2', solver='saga', random_state=21, fit_intercept=False, max_iter=5000)
skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    lr_l2.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, lr_l2.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was depreca

Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: total: 766 ms
Wall time: 769 ms


c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [221]:
%%time
svc_base = SVC(probability=True, kernel='linear', random_state=21)

skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    svc_base.fit(X_train_fold, y_train_fold)

    train_acc = accuracy_score(y_train_fold, svc_base.predict(X_train_fold))
    valid_acc = accuracy_score(y_valid_fold, svc_base.predict(X_valid_fold))

    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: total: 2.16 s
Wall time: 2.22 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [222]:
%%time
svc_c01 = SVC(C=0.1, kernel='linear', probability=True, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    svc_c01.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, svc_c01.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

Average accuracy on crossval is 0.56230
Std is 0.02177
CPU times: total: 1.98 s
Wall time: 1.99 s


In [223]:
%%time
svc_c10 = SVC(C=10, kernel='linear', probability=True, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    svc_c10.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, svc_c10.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

Average accuracy on crossval is 0.72771
Std is 0.04417
CPU times: total: 2.95 s
Wall time: 3.01 s


In [224]:
%%time
svc_c100 = SVC(C=100, kernel='linear', probability=True, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    svc_c100.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, svc_c100.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.75589
Std is 0.03550
CPU times: total: 11.4 s
Wall time: 11.7 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [225]:
dt_base = DecisionTreeClassifier(max_depth=10, random_state=21)
skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    dt_base.fit(X_train_fold, y_train_fold)

    train_acc = accuracy_score(y_train_fold, dt_base.predict(X_train_fold))
    valid_acc = accuracy_score(y_valid_fold, dt_base.predict(X_valid_fold))
    
    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.81039   |   valid -  0.74815
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.77037
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.74627
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72625
Std is 0.03635


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [226]:
%%time
dt_d5 = DecisionTreeClassifier(max_depth=5, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    dt_d5.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, dt_d5.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.54301
Std is 0.02423
CPU times: total: 46.9 ms
Wall time: 48.8 ms


In [227]:
%%time
dt_d50 = DecisionTreeClassifier(max_depth=50, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    dt_d50.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, dt_d50.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.88501
Std is 0.02105
CPU times: total: 46.9 ms
Wall time: 61.7 ms


### bonus model

In [228]:
%%time
dt_bonus = DecisionTreeClassifier(max_depth=21, min_samples_leaf=1, random_state=21)
skf = StratifiedKFold(n_splits=10)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    dt_bonus.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, dt_bonus.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.88501
Std is 0.01796
CPU times: total: 62.5 ms
Wall time: 56.7 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [229]:
rf_base = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
skf = StratifiedKFold(n_splits=10)

valid_scores = []
for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    rf_base.fit(X_train_fold, y_train_fold)

    train_acc = accuracy_score(y_train_fold, rf_base.predict(X_train_fold))
    valid_acc = accuracy_score(y_valid_fold, rf_base.predict(X_valid_fold))

    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.96373   |   valid -  0.87407
train -  0.97032   |   valid -  0.91111
train -  0.96867   |   valid -  0.88889
train -  0.97279   |   valid -  0.91111
train -  0.96785   |   valid -  0.91111
train -  0.96620   |   valid -  0.85185
train -  0.96867   |   valid -  0.91111
train -  0.96702   |   valid -  0.85185
train -  0.97199   |   valid -  0.88060
train -  0.96458   |   valid -  0.85075
Average accuracy on crossval is 0.88425
Std is 0.02499


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [230]:
%%time
rf_n100 = RandomForestClassifier(n_estimators=100, max_depth=14, random_state=21)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    rf_n100.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, rf_n100.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

Average accuracy on crossval is 0.88872
Std is 0.03114
CPU times: total: 1.45 s
Wall time: 1.49 s


In [231]:
%%time
rf_d25 = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    rf_d25.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, rf_d25.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.91691
Std is 0.02984
CPU times: total: 1.72 s
Wall time: 1.74 s


### bonus model

In [232]:
%%time
rf_heavy = RandomForestClassifier(
    n_estimators=219, 
    max_depth=30, 
    min_samples_split=3, 
    random_state=21
)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    rf_heavy.fit(X_train_fold, y_train_fold)
    valid_scores.append(accuracy_score(y_valid_fold, rf_heavy.predict(X_valid_fold)))

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

Average accuracy on crossval is 0.92433
Std is 0.02669
CPU times: total: 3.62 s
Wall time: 3.72 s


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [233]:
final_model = RandomForestClassifier(
    n_estimators=219, 
    max_depth=30, 
    min_samples_split=3, 
    random_state=21
)

In [234]:
final_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",219
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [235]:
y_pred = final_model.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)

In [236]:
final_acc

0.9349112426035503

### errors analyze

In [237]:
cm = confusion_matrix(y_test, y_pred)

In [238]:
class_error_rates = (cm.sum(axis=1) - cm.diagonal()) / cm.sum(axis=1)

In [239]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
error_df = pd.DataFrame({'Day': days, 'Error Rate %': class_error_rates * 100})

In [240]:
error_df

,Day,Error Rate %
0,Mon,25.925926
1,Tue,7.272727
2,Wed,6.666667
3,Thu,2.500000
4,Fri,14.285714
5,Sat,5.555556
6,Sun,1.408451


model makes the most errors at monday

In [241]:
joblib.dump(final_model, '../data/final_model.joblib')

['../data/final_model.joblib']